# Lab 10 · Multimodal retrieval

**Day 3 · S16, lab 3 of 3** · Budget: 25 min of the 30 min slot · Runs on: Colab with a T4 GPU (a laptop works, slower)

This morning's pipeline retrieves across 74 documents and answers questions about them. It cannot answer one question about the drawings and work order forms from labs 08 and 09, because a PNG holds no text to retrieve.

This lab joins the two halves. The pattern is the one most enterprises land on:

**extract once, offline → index the extraction as text → retrieve as text → look at the pixels again only where it matters.**

| Stage | Cost | How often it runs |
|---|---|---|
| Vision extraction (labs 08, 09) | seconds per image, on a GPU or an API | once per image, when the image is filed |
| Text retrieval (lab 07) | milliseconds | every question |
| Vision verification | one more image call | only on the answers that carry risk |

You will see four things, in this order: retrieval that cannot see images, retrieval that can, retrieval that is only as good as the extraction behind it, and an answer that is confidently wrong because the extraction invented a value.

### What this lab loads from the other labs

| From | What | If you skipped it |
|---|---|---|
| Lab 07 | `artifacts/rag_index/` — the text index, 342 chunks | it is committed in the repo, so the notebook just loads it |
| Labs 08, 09 | `outputs/08/`, `outputs/09/` — what each model read off each image | the lab falls back to the ground truth and says so in its output |

## 1. Setup

The next three cells find the lab folder, pick the models and load this morning's index. The answering model is the same self-hosted or vendor model you used in labs 08 and 09: this lab compares *extractions*, not answering models, so one is enough.

In [3]:
# Setup: find the lab folder, detect the runtime, install pinned packages on Colab.
import os
import re
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = os.environ.get("LAB_REPO_URL", "")  # Colab: the course repo URL, once it is published


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "vision_client.py").exists():
            return p


ROOT = find_root()
if ROOT is None and IN_COLAB and REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    ROOT = Path("/content/lab")
if ROOT is None:
    raise RuntimeError("Lab folder not found. Open this notebook from inside it, or set LAB_REPO_URL on Colab.")
if IN_COLAB:  # Colab already ships torch and sentence-transformers; rank-bm25 is the lexical half of the index
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "openai==3.0.0", "python-dotenv==1.1.0", "rank-bm25==0.2.2"], check=True)
sys.path.insert(0, str(ROOT / "scripts"))
print("lab folder:", ROOT, "| runtime:", "Colab" if IN_COLAB else "local")

lab folder: /Users/drpreetyrai./aiguru | runtime: local


In [4]:
import json

import numpy as np
import pandas as pd
from vision_client import ask, ensure_ollama, load_openai_key, prebaked_models, run_batch, self_hosted, vendor_api

pd.set_option("display.max_colwidth", 90)

RUN_MODE = os.environ.get("LAB_RUN_MODE", "live")  # "live" calls the models, "prebaked" replays a saved run
LAB = "10"
OUT = ROOT / "outputs" / LAB  # the known-bad pair goes to outputs/10_kb_*, beside it
PREBAKED = Path(os.environ.get("LAB_PREBAKED_DIR", ROOT / "facilitator" / "prebaked_outputs")) / LAB

MODELS = []
if RUN_MODE == "live":
    try:
        ensure_ollama()
        MODELS.append(self_hosted())
    except Exception as e:
        print("self-hosted model unavailable:", e)
    if load_openai_key(ROOT):
        MODELS.append(vendor_api())
    else:
        print("vendor API unavailable: no OPENAI_API_KEY in Colab secrets or .env")
if not MODELS:
    MODELS = prebaked_models(PREBAKED)
    print("using prebaked outputs from", PREBAKED)
assert MODELS, f"No live model and no prebaked outputs in {PREBAKED}. Ask the facilitator."
ANSWER_MODEL = MODELS[0]
print("models:", [m.label for m in MODELS], "| answering with:", ANSWER_MODEL.label)

self-hosted model unavailable: Ollama is not installed. See setup/ollama_setup.md or https://ollama.com/download
models: ['vendor API: gpt-4.1-mini'] | answering with: vendor API: gpt-4.1-mini


In [5]:
from rag_index import Chunk, RagIndex

TEXT = RagIndex.load(ROOT / "artifacts" / "rag_index")
MANIFEST = pd.read_csv(ROOT / "corpus" / "images" / "manifest.csv")
TRUTH = ROOT / "data" / "eval" / "image_ground_truth"

print(f"text index : {len(TEXT.chunks)} chunks from {TEXT.manifest['documents']} documents")
print(f"             embeddings {TEXT.manifest['embed_model']}, rerank {TEXT.manifest['rerank_model']}")
print(f"images     : {len(MANIFEST)} from labs 08 and 09")
print("The embedder and the reranker download on the first search, which takes about a minute.")

text index : 342 chunks from 74 documents
             embeddings BAAI/bge-small-en-v1.5, rerank cross-encoder/ms-marco-MiniLM-L-12-v2
images     : 20 from labs 08 and 09
The embedder and the reranker download on the first search, which takes about a minute.


## 2. The question set

Ten questions across the two source types. `text_only` is the control: it was answerable this morning, and it has to stay answerable after you put images into the index.

| Category | What it needs |
|---|---|
| `text_only` | a document, nothing else |
| `image_only` | one drawing or one form |
| `cross_modal` | a reading from a form **and** a limit from a manual |
| `blank_field` | a form field that is blank; the only correct answer says so |
| `unanswerable` | neither; the model should refuse |

In [6]:
# @title Materialise the S16 question set (skip-safe: never overwrites a committed file) { display-mode: "form" }
# data/eval/multimodal_questions.jsonl : the question, where its answer lives, and how it is scored.
# evidence_match scores retrieval (did the answer text reach the model), must_match scores the answer.
QUESTIONS_PATH = ROOT / "data" / "eval" / "multimodal_questions.jsonl"
QUESTIONS = [
    # --- control: the answer is in the text corpus, exactly as it was this morning -----------------
    dict(id="M01", category="text_only",
         question="What is the frame vibration alarm setpoint for compressor K-301?",
         gold_sources=["manuals/MAN-K-301.md"], answerable=True,
         reference="The K-301 manual sets the frame vibration alarm at 9.0 mm/s and the trip at 14.0 mm/s.",
         must_match=[r"\b9(\.0)?\b"], must_not_match=[], evidence_match=[r"9\.0"]),

    # --- the answer exists only inside an image ----------------------------------------------------
    dict(id="M02", category="image_only",
         question="Which protocol does the Integration Hub use to send work orders to the ERP?",
         gold_sources=["images/dia_01.png", "images/dia_04.png"], answerable=True,
         reference="The integration diagrams show the Integration Hub reaching the ERP (PM module) over SOAP.",
         must_match=[r"soap"], must_not_match=[], evidence_match=[r"SOAP"]),
    dict(id="M03", category="image_only",
         question="In the alarm and incident flow, what carries events from the event bus to on-call paging?",
         gold_sources=["images/dia_06.png"], answerable=True,
         reference="The alarm and incident flow diagram shows a webhook from the Event Bus to On-call Paging.",
         must_match=[r"webhook"], must_not_match=[], evidence_match=[r"Webhook"]),
    dict(id="M04", category="image_only",
         question="What vibration reading is recorded on work order WO-2026-04817?",
         gold_sources=["images/wo_01.png"], answerable=True,
         reference="Work order WO-2026-04817 records a vibration of 4.6 mm/s.",
         must_match=[r"4\.6"], must_not_match=[], evidence_match=[r"4\.6"]),
    dict(id="M05", category="image_only",
         question="Who is work order WO-2026-05033 assigned to?",
         gold_sources=["images/wo_03.png"], answerable=True,
         reference="Work order WO-2026-05033 is assigned to A. Khan.",
         must_match=[r"khan"], must_not_match=[], evidence_match=[r"Khan"]),

    # --- one fact from a document, one from an image ------------------------------------------------
    dict(id="M06", category="cross_modal",
         question=("Work order WO-2026-04817 records a vibration reading for K-301. "
                   "Is that reading above the alarm setpoint in the K-301 manual?"),
         gold_sources=["images/wo_01.png", "manuals/MAN-K-301.md"], answerable=True,
         reference="The work order records 4.6 mm/s. The K-301 alarm is 9.0 mm/s, so the reading is below the alarm.",
         must_match=[r"4\.6", r"\b9(\.0)?\b"], must_not_match=[r"(above|exceeds|over|higher than) the alarm"],
         evidence_match=[r"4\.6", r"9\.0"]),
    dict(id="M07", category="cross_modal",
         question=("The inspection work order on E-401 records a differential pressure. "
                   "Does the E-401 manual require the bundle to be cleaned at that value?"),
         gold_sources=["images/wo_03.png", "manuals/MAN-E-401.md"], answerable=True,
         reference=("The work order records 0.9 bar. The manual calls for cleaning above 1.2 bar sustained for "
                    "more than 7 days, so cleaning is not required."),
         must_match=[r"0\.9", r"1\.2"], must_not_match=[], evidence_match=[r"0\.9", r"1\.2"]),
    dict(id="M08", category="cross_modal",
         question=("Work order WO-2026-04902 records a vibration reading on P-101A. "
                   "How does it compare with the alarm setpoint in the P-101A manual?"),
         gold_sources=["images/wo_02.png", "manuals/MAN-P-101A.md"], answerable=True,
         reference="The work order records 2.1 mm/s against a bearing vibration alarm of 7.1 mm/s, well below it.",
         must_match=[r"2\.1", r"7\.1"], must_not_match=[r"(above|exceeds|over|higher than) the alarm"],
         evidence_match=[r"2\.1", r"7\.1"]),

    # --- the field is blank on the form: the only correct answer says so ----------------------------
    dict(id="M09", category="blank_field",
         question="What permit to work number is recorded on work order WO-2026-05111?",
         gold_sources=["images/wo_10.png"], answerable=True,
         reference="The permit to work field on WO-2026-05111 is blank. No permit number is recorded.",
         must_match=[r"(blank|empty|no permit|not recorded|none|left|does not|is not|isn.?t|missing|null)"],
         must_not_match=[r"PTW-\d"], evidence_match=[r"WO-2026-05111"]),

    # --- in neither the documents nor the images ----------------------------------------------------
    dict(id="M10", category="unanswerable",
         question="Which protocol connects the Integration Hub to the SAP Ariba procurement portal?",
         gold_sources=[], answerable=False,
         reference="No SGP document or drawing mentions SAP Ariba. The right answer is that it is not covered.",
         must_match=[], must_not_match=[r"\b(REST|SOAP|Kafka|MQTT|OData|JDBC|SFTP|OAuth2?)\b"],
         evidence_match=[]),
]
if not QUESTIONS_PATH.exists():
    QUESTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)
    QUESTIONS_PATH.write_text("\n".join(json.dumps(q) for q in QUESTIONS) + "\n", encoding="utf-8")
    print("wrote", QUESTIONS_PATH)

EVAL = [json.loads(line) for line in QUESTIONS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
pd.DataFrame(EVAL)[["id", "category", "question"]]

,id,category,question
0,M01,text_only,What is the frame vibration alarm setpoint for compressor K-301?
1,M02,image_only,Which protocol does the Integration Hub use to send work orders to the ERP?
2,M03,image_only,"In the alarm and incident flow, what carries events from the event bus to on-call paging?"
3,M04,image_only,What vibration reading is recorded on work order WO-2026-04817?
4,M05,image_only,Who is work order WO-2026-05033 assigned to?
5,M06,cross_modal,Work order WO-2026-04817 records a vibration reading for K-301. Is that reading above ...
6,M07,cross_modal,The inspection work order on E-401 records a differential pressure. Does the E-401 man...
7,M08,cross_modal,Work order WO-2026-04902 records a vibration reading on P-101A. How does it compare wi...
8,M09,blank_field,What permit to work number is recorded on work order WO-2026-05111?
9,M10,unanswerable,Which protocol connects the Integration Hub to the SAP Ariba procurement portal?


In [7]:
# The measurements, defined once so every retrieval configuration is compared on identical terms.
def evidence(item: dict, hits: list) -> float:
    """1 if the chunks retrieved from the right source actually contain the answer text."""
    gold = set(item["gold_sources"])
    if not gold:
        return np.nan  # nothing to find: the unanswerable question is scored on its answer instead
    found = "\n".join(h.text for h in hits if h.source in gold)
    return float(all(re.search(p, found, re.I) for p in item["evidence_match"]))


def measure(retrieve, name: str) -> pd.DataFrame:
    rows = []
    for item in EVAL:
        hits = retrieve(item["question"])
        rows.append({"retrieval": name, "id": item["id"], "category": item["category"],
                     "evidence": evidence(item, hits),
                     "sources": [h.source.split("/")[-1] for h in hits]})
    return pd.DataFrame(rows)


def compare(*runs: pd.DataFrame) -> pd.DataFrame:
    """Answer text retrieved, by category. 1.00 means every question in that row found its evidence."""
    table = pd.concat(runs).pivot_table(index="category", columns="retrieval", values="evidence", aggfunc="mean")
    return table[[r["retrieval"].iat[0] for r in runs]].round(2)

## 3. The gap

Run this morning's index against all ten questions. The `text_only` control passes. Everything that lives in an image fails, and it fails quietly: the retriever returns confident-looking chunks about the right equipment, none of which contain the answer.

In [8]:
TEXT_ONLY = measure(lambda q: TEXT.search(q, k=6), "text only")
compare(TEXT_ONLY)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7636.58it/s]


retrieval,text only
category,
blank_field,0.0
cross_modal,0.0
image_only,0.0
text_only,1.0


In [9]:
item = next(i for i in EVAL if i["id"] == "M04")
print(item["question"], "\n")
for hit in TEXT.search(item["question"], k=4):
    print(f"  {hit.score:6.2f}  {hit.source:32s} {hit.section[:45]}")
print("\nThe answer is on a form the index has never seen. Nothing in the result says so.")

What vibration reading is recorded on work order WO-2026-04817? 

    4.15  maintenance/WO-2026-0230.md      Follow-up
    3.83  maintenance/WO-2026-0230.md      Problem description
    3.47  maintenance/WO-2026-0281.md      Problem description
    3.19  maintenance/LOG-2026-06-12-D.md  Maintenance

The answer is on a form the index has never seen. Nothing in the result says so.


## 4. Three ways to put an image into a text index

| Approach | What gets indexed | Where it fails |
|---|---|---|
| Filename and caption | `wo_01.png`, "work order form" | every form retrieves identically; nothing inside is searchable |
| Ask the model to describe the image | a paragraph of prose | prose loses tags and numbers or mangles them: `K-301` becomes "the compressor". BM25, which is what finds tags, has nothing to match |
| **Index the structured extraction** | the JSON from labs 08 and 09, rendered as a passage | the extraction can be wrong, and then the index is wrong. Section 7 |

The third wins for enterprise documents because a tag, a permit number and a work order ID survive it exactly. It also gives you something a caption never does: the same JSON goes into the CMMS, and the passage is a second use of it, not a second extraction.

Two details in the renderer below carry lab 09's lesson forward:

- a blank field is written out as `(blank on the form)` rather than dropped. A field that disappears is indistinguishable from a field nobody asked for.
- the passage names the image it came from and the model that read it, so a citation points at something a person can open, and a re-extraction knows what to rebuild.

In [10]:
BLANK = "(blank on the form)"


def _value(v):
    return BLANK if v is None or v == "" else v


def diagram_passage(rec: dict) -> str:
    lines = [f"Integration diagram: {rec.get('title') or 'untitled'}",
             "Systems: " + "; ".join(rec.get("nodes") or []),
             "Connections:"]
    for e in rec.get("edges") or []:
        label = f" ({e['label']})" if e.get("label") else ""
        lines.append(f"- {e.get('from')} -> {e.get('to')}{label}")
    return "\n".join(lines)


def form_passage(rec: dict) -> str:
    lines = [f"Maintenance work order {_value(rec.get('work_order_id'))}",
             f"Site: {_value(rec.get('site'))}",
             f"Equipment tag: {_value(rec.get('equipment_tag'))} ({_value(rec.get('equipment_description'))})",
             f"Date raised: {_value(rec.get('date_raised'))}",
             f"Priority: {_value(rec.get('priority'))}   Work type: {_value(rec.get('work_type'))}",
             f"Requested by: {_value(rec.get('requested_by'))}   Assigned to: {_value(rec.get('assigned_to'))}",
             f"Problem description: {_value(rec.get('problem_description'))}",
             "Readings:"]
    for r in rec.get("readings") or []:
        lines.append(f"- {r.get('parameter')}: {_value(r.get('value'))} {r.get('unit') or ''}".rstrip())
    lines += [f"Permit to work no: {_value(rec.get('permit_number'))}",
              f"Supervisor sign-off date: {_value(rec.get('supervisor_signoff_date'))}"]
    return "\n".join(lines)


def image_chunks(extractions: dict, read_by: str) -> list:
    """One chunk per image, in the same Chunk shape as a document chunk so one index holds both."""
    out = []
    for row in MANIFEST.itertuples():
        rec = extractions.get(row.item_id)
        if not rec:
            continue
        body = diagram_passage(rec) if row.kind == "diagram" else form_passage(rec)
        kind = "Integration diagram" if row.kind == "diagram" else "Work order form"
        out.append(Chunk(chunk_id=f"images/{row.item_id}.png#0", source=f"images/{row.item_id}.png",
                         doc_id=row.item_id, revision=1, status="current",
                         title=f"{kind} {row.item_id}", section="extracted",
                         text=f"Read from image {row.item_id}.png by {read_by} [tier {row.tier}]\n{body}"))
    return out

In [11]:
# Tiers 1 and 2, plus the two known-bad images. The degraded copies of the same originals are section 11.
INDEXED = ["dia_01", "dia_02", "dia_03", "dia_04", "dia_05", "dia_06", "dia_10",
           "wo_01", "wo_02", "wo_03", "wo_10"]

GOLD = {i: json.loads((TRUTH / f"{i}.json").read_text()) for i in INDEXED}
print(image_chunks({"wo_10": GOLD["wo_10"]}, "ground truth")[0].text)

Read from image wo_10.png by ground truth [tier known-bad]
Maintenance work order WO-2026-05111
Site: Sabkha Gas Plant
Equipment tag: V-203 (Test separator)
Date raised: 2026-09-09
Priority: P2   Work type: Corrective
Requested by: H. Al-Balushi   Assigned to: R. Menon
Problem description: Level gauge LG-203 reading erratic. Replace gauge glass.
Readings:
- Operating pressure: 12.4 bar
- Liquid level: (blank on the form) %
- Temperature: 41 °C
Permit to work no: (blank on the form)
Supervisor sign-off date: (blank on the form)


Now load what the models actually read in labs 08 and 09. That is the difference between the index you would have if extraction were perfect and the index you really have.

In [12]:
def extraction_models() -> list:
    names = set()
    for lab in ("08", "09"):
        for base in (ROOT / "outputs" / lab, ROOT / "facilitator" / "prebaked_outputs" / lab):
            if base.exists():
                names |= {p.name for p in base.iterdir() if p.is_dir()}
    return sorted(names)


def read_extractions(model_name: str, item_ids: list) -> dict:
    """What this model returned for each image in labs 08 and 09. Live outputs first, then prebaked."""
    found = {}
    for item_id in item_ids:
        for lab in ("08", "09"):
            for base in (ROOT / "outputs" / lab, ROOT / "facilitator" / "prebaked_outputs" / lab):
                rec_path = base / model_name / f"{item_id}.json"
                if rec_path.exists():
                    rec = json.loads(rec_path.read_text())
                    if rec.get("output"):
                        found[item_id] = rec["output"]
                        break
            if item_id in found:
                break
    return found


EXTRACTOR, EXTRACTED = None, {}
for name in extraction_models():
    found = read_extractions(name, INDEXED)
    if len(found) > len(EXTRACTED):
        EXTRACTOR, EXTRACTED = name, found

if EXTRACTOR:
    print(f"indexing the extractions from {EXTRACTOR}: {len(EXTRACTED)} of {len(INDEXED)} images")
else:
    print("No extractions found from labs 08 or 09, so sections 7 to 10 fall back to the ground truth.")
    print("That makes the pipeline look better than it is. Run labs 08 and 09 first if you can.")

indexing the extractions from openai-gpt-4.1-mini: 11 of 11 images


In [13]:
GOLD_CHUNKS = image_chunks(GOLD, "ground truth")
READ_CHUNKS = image_chunks(EXTRACTED, EXTRACTOR) if EXTRACTOR else GOLD_CHUNKS

PERFECT = TEXT.copy().add(GOLD_CHUNKS)   # the index you would have if extraction never made a mistake
ACTUAL = TEXT.copy().add(READ_CHUNKS)    # the index you really have
print(f"{len(TEXT.chunks)} document chunks + {len(GOLD_CHUNKS)} image chunks = {len(PERFECT.chunks)}")

342 document chunks + 11 image chunks = 353


## 5. One index, one ranking

Both source types are in one index now, and one ranking decides what reaches the model. Compare with section 3.

In [14]:
MIXED = measure(lambda q: PERFECT.search(q, k=6), "mixed, k=6")
compare(TEXT_ONLY, MIXED)

retrieval,text only,"mixed, k=6"
category,,
blank_field,0.0,1.00
cross_modal,0.0,0.33
image_only,0.0,0.75
text_only,1.0,1.00


In [15]:
MIXED[MIXED.evidence == 0][["id", "category", "sources"]]

,id,category,sources
4,M05,image_only,"[wo_10.png, WO-2026-0257.md, WO-2026-0244.md, WO-2026-0201.md, WO-2026-0250.md, WO-202..."
5,M06,cross_modal,"[wo_01.png, wo_02.png, WO-2026-0234.md, WO-2026-0234.md, INSP-2026-022.md, RCA-2026-00..."
7,M08,cross_modal,"[wo_02.png, wo_01.png, WO-2026-0142.md, WO-2026-0281.md, INSP-2026-022.md, WO-2026-011..."


Two different failures are in that list and they have one cause.

**The cross-modal questions.** Each needs a reading from a form *and* a limit from a manual. One query, one ranking, six slots: the six best-matching chunks are all work-order-shaped, because the question is phrased like a work order. The manual never arrives.

**`M05`**, a plain lookup by work order number. Thirty-odd work orders in the text corpus look almost exactly like the form, so the one chunk holding the answer has to beat all of them on a single score just to reach the rerank.

Adding a source type to an index does not only add rows. It adds competition, and the retrieval settings that were right this morning are not right any more.

## 6. Give each source type its own slots

The fix is not a better score. It is to stop making unlike things compete: take the top 2 from each source type and concatenate. Six chunks either way, so the model reads the same amount of context.

This is worth doing at OQ for a reason beyond the score. A document class is also a permission boundary and a lifecycle: manuals are revision-controlled, work orders are closed, drawings are re-issued. Retrieving them separately is how you come to filter them separately.

In [16]:
SOURCE_TYPES = {"specifications": ("manuals/", "hse/"), "events": ("maintenance/",), "drawings": ("images/",)}


def route(index: RagIndex) -> dict:
    """Split one index into one sub-index per source type. Chunks and embeddings are already in memory."""
    subs = {}
    for name, prefixes in SOURCE_TYPES.items():
        keep = [i for i, c in enumerate(index.chunks) if c.source.startswith(prefixes)]
        subs[name] = RagIndex([index.chunks[i] for i in keep], index.embeddings[keep], index.manifest)
    return subs


def routed_search(subs: dict, question: str, k: int = 2) -> list:
    return [hit for sub in subs.values() for hit in sub.search(question, k=k)]


SUBS_PERFECT = route(PERFECT)
ROUTED = measure(lambda q: routed_search(SUBS_PERFECT, q), "per source type, 2+2+2")
compare(TEXT_ONLY, MIXED, ROUTED)

retrieval,text only,"mixed, k=6","per source type, 2+2+2"
category,,,
blank_field,0.0,1.00,1.0
cross_modal,0.0,0.33,1.0
image_only,0.0,0.75,1.0
text_only,1.0,1.00,1.0


In [17]:
item = next(i for i in EVAL if i["id"] == "M06")
print(item["question"], "\n")
for hit in routed_search(SUBS_PERFECT, item["question"]):
    print(f"  {hit.source:32s} {hit.section[:48]}")
print("\nThe reading comes from the form, the setpoint from the manual. Neither had to outrank the other.")

Work order WO-2026-04817 records a vibration reading for K-301. Is that reading above the alarm setpoint in the K-301 manual? 

  manuals/MAN-K-301.md             5. Operating limits and alarms
  manuals/MAN-K-301.md             1. Purpose and scope
  maintenance/WO-2026-0281.md      Problem description
  maintenance/WO-2026-0201.md      Work performed
  images/wo_01.png                 extracted
  images/wo_02.png                 extracted

The reading comes from the form, the setpoint from the manual. Neither had to outrank the other.


## 7. Retrieval is only as good as the extraction

`PERFECT` was built from the ground truth: every box, arrow and field read correctly. `ACTUAL` was built from what the vision model actually returned this afternoon. Same documents, same retriever, same questions. The only difference is who read the images.

In [18]:
SUBS_ACTUAL = route(ACTUAL)
ROUTED_ACTUAL = measure(lambda q: routed_search(SUBS_ACTUAL, q), f"read by {EXTRACTOR or 'ground truth'}")
compare(ROUTED, ROUTED_ACTUAL)

retrieval,"per source type, 2+2+2",read by openai-gpt-4.1-mini
category,,
blank_field,1.0,1.0
cross_modal,1.0,1.0
image_only,1.0,1.0
text_only,1.0,1.0


Those two columns may well be identical, and that is not reassurance. Ten questions cannot touch every field in eleven images. What it means is that this afternoon's extraction errors happen to sit in fields nobody asked about, which is luck, not safety.

So score the chunks themselves. `scripts/score_extraction.py` is the same scorer you ran in labs 08 and 09; point it at the images that went into the index and it tells you which chunks are carrying a wrong value right now.

In [19]:
from score_extraction import score_dir


def pred_root(lab: str) -> Path:
    live = ROOT / "outputs" / lab
    return live if live.exists() else ROOT / "facilitator" / "prebaked_outputs" / lab


if EXTRACTOR:
    quality = pd.concat([score_dir(pred_root("08"), TRUTH, MANIFEST, "diagram"),
                         score_dir(pred_root("09"), TRUTH, MANIFEST, "form")])
    quality = quality[(quality.model == EXTRACTOR) & quality.item_id.isin(INDEXED)]
    print(f"{EXTRACTOR} scored {quality.score.mean():.2f} on the {len(quality)} images in this index")
    wrong = quality[quality.score < 1].assign(first_error=lambda d: d.errors.str[0])
else:
    wrong = pd.DataFrame(columns=["item_id", "tier", "score", "first_error"])
    print("built from the ground truth, so every chunk is correct by construction")
wrong[["item_id", "tier", "score", "first_error"]]

openai-gpt-4.1-mini scored 0.99 on the 11 images in this index


,item_id,tier,score,first_error
5,dia_06,2,0.938,reversed edge: Data Lake -> ITSM Platform


Every row above is a passage in the index that says something the image does not, and it will be retrieved and cited exactly as confidently as a correct one. That is worth being precise about. A retrieval bug shows up as "I couldn't find it", which people report. An extraction error shows up as a confident answer with a citation, which people do not report. You find out when somebody acts on it.

Two consequences for the capstone:

1. **The tier scores from labs 08 and 09 are retrieval scores too.** An image class that extracts at 0.6 does not belong in the index yet.
2. **Re-extraction is a migration.** Change the vision model and every chunk derived from an image is stale. Keep the image id and the extracting model in the chunk, as the renderer does, so you can find them and rebuild them.

## 8. Answering, with citations

The prompt is the grounded prompt from lab 07 with two additions: passages say whether they came from a document or from an image, and `(blank on the form)` is given an explicit meaning. Without that second rule the model treats a blank as a gap to fill, which is the failure lab 09 was built around.

In [20]:
GROUNDED = """\
You answer questions for staff of the Sabkha Gas Plant (SGP) using only the passages below.
Some passages are documents. Some were read off a drawing or a scanned form by a vision model, and say so.

Rules:
1. Use only what the passages state. If they do not answer the question, reply exactly:
   "I don't know based on the SGP documents and drawings." Never guess.
2. "(blank on the form)" means the form does not record that field. Say that it is blank.
   Never supply a value for it, however plausible.
3. Cite the source of every fact in square brackets, exactly as the passage is labelled.
4. Answer in at most three sentences.

Passages:
{context}

Question: {question}
Answer:"""


def build_prompt(question: str, hits: list) -> str:
    context = "\n\n".join(f"[{h.source}]\n{h.text}" for h in hits)
    return GROUNDED.format(context=context, question=question)


demo = next(i for i in EVAL if i["id"] == "M06")
hits = routed_search(SUBS_ACTUAL, demo["question"])
pair = [hits[0], next(h for h in hits if h.source.startswith("images/"))]
print(f"The model sees {len(hits)} passages. These two carry the answer:\n")
print(build_prompt(demo["question"], pair))

The model sees 6 passages. These two carry the answer:

You answer questions for staff of the Sabkha Gas Plant (SGP) using only the passages below.
Some passages are documents. Some were read off a drawing or a scanned form by a vision model, and say so.

Rules:
1. Use only what the passages state. If they do not answer the question, reply exactly:
   "I don't know based on the SGP documents and drawings." Never guess.
2. "(blank on the form)" means the form does not record that field. Say that it is blank.
   Never supply a value for it, however plausible.
3. Cite the source of every fact in square brackets, exactly as the passage is labelled.
4. Answer in at most three sentences.

Passages:
[manuals/MAN-K-301.md]
Export Gas Compressor K-301 - Operation and Maintenance Manual [MAN-K-301 rev 2, current]
Section: 5. Operating limits and alarms

Alarms are annunciated on the DCS operator graphics. A trip stops the driver and requires a field check before restart.

| Measurement | Alarm |

In [21]:
ABSTAIN = re.compile(r"i don.?t know|do not know|not (mentioned|found|specified|stated|provided|available|covered)"
                     r"|no (information|record|mention)|cannot (find|determine|answer)", re.I)


def score_answer(item: dict, text: str) -> float:
    trap = any(re.search(p, text, re.I) for p in item["must_not_match"])
    if not item["answerable"]:
        return float(bool(ABSTAIN.search(text)) and not trap)
    return float(all(re.search(p, text, re.I) for p in item["must_match"]) and not trap)


def answer_all(subs: dict, run: str) -> pd.DataFrame:
    """Retrieve, answer and score every question. Cached in outputs/<run>/, so a re-run costs nothing."""
    retrieved = {item["id"]: routed_search(subs, item["question"]) for item in EVAL}
    jobs = [{"item_id": item["id"], "prompt": build_prompt(item["question"], retrieved[item["id"]])}
            for item in EVAL]
    recs = run_batch(ANSWER_MODEL, jobs, out_dir=ROOT / "outputs" / run,
                     prebaked_dir=PREBAKED.parent / run,
                     workers=4 if ANSWER_MODEL.backend == "openai" else 1)
    answers = {r["item_id"]: (r["output"] or "") for r in recs}
    return pd.DataFrame([{"id": item["id"], "category": item["category"],
                          "evidence": evidence(item, retrieved[item["id"]]),
                          "correct": score_answer(item, answers[item["id"]]),
                          "answer": " ".join(answers[item["id"]].split())} for item in EVAL])


RESULTS = answer_all(SUBS_ACTUAL, LAB)
RESULTS

  M03: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked  M04: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked

  M01: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
  M02: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
  M05: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
  M06: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
  M07: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
  M08: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
  M09: live call

,id,category,evidence,correct,answer
0,M01,text_only,1.0,1.0,The frame vibration alarm setpoint for compressor K-301 is 9.0 mm/s. The trip setpoint...
1,M02,image_only,1.0,1.0,The Integration Hub uses the SOAP protocol to send work orders to the ERP (PM module) ...
2,M03,image_only,1.0,1.0,Events are carried from the event bus to on-call paging via a webhook [images/dia_06.p...
3,M04,image_only,1.0,1.0,The vibration reading recorded on work order WO-2026-04817 is 4.6 mm/s [images/wo_01.p...
4,M05,image_only,1.0,1.0,Work order WO-2026-05033 is assigned to A. Khan [images/wo_03.png].
5,M06,cross_modal,1.0,0.0,"The vibration reading for K-301 in work order WO-2026-04817 is 4.6 mm/s, which is belo..."
6,M07,cross_modal,1.0,1.0,The inspection work order WO-2026-05033 records a differential pressure of 0.9 bar on ...
7,M08,cross_modal,1.0,1.0,The vibration reading recorded on work order WO-2026-04902 for P-101A is 2.1 mm/s. Thi...
8,M09,blank_field,1.0,1.0,The permit to work number recorded on work order WO-2026-05111 is blank on the form. [...
9,M10,unanswerable,NaN,1.0,I don't know based on the SGP documents and drawings.


In [22]:
RESULTS.groupby("category")[["evidence", "correct"]].mean().round(2)

,evidence,correct
category,,
blank_field,1.0,1.00
cross_modal,1.0,0.67
image_only,1.0,1.00
text_only,1.0,1.00
unanswerable,NaN,1.00


Read the rows where `evidence` is 1 and `correct` is 0: the answer text was in front of the model and it still got it wrong. That is a generation failure and the prompt is the fix. Where `evidence` is 0, retrieval never delivered the answer and no prompt will save it. Same split as this morning, and it is still the first question to ask of any failure.

## 9. The known-bad case, one step further

`wo_10` has three blank fields. Lab 09 asked whether a model invents values for them. This asks what happens next, once the extraction is in the index and nobody is looking at the form any more.

In [23]:
def permit_line(chunks: list) -> str:
    chunk = next((c for c in chunks if c.source == "images/wo_10.png"), None)
    if chunk is None:
        return "not indexed"
    return next((ln for ln in chunk.text.splitlines() if ln.startswith("Permit to work")), "no permit line")


print(f"{'ground truth':>22s} :", permit_line(GOLD_CHUNKS))
print(f"{(EXTRACTOR or 'ground truth'):>22s} :", permit_line(READ_CHUNKS))

          ground truth : Permit to work no: (blank on the form)
   openai-gpt-4.1-mini : Permit to work no: (blank on the form)


In [24]:
item = next(i for i in EVAL if i["id"] == "M09")
print(item["question"], "\n")
KB = {}
for tag, subs in [("perfect", SUBS_PERFECT), ("actual", SUBS_ACTUAL)]:
    hits = routed_search(subs, item["question"])
    rec = run_batch(ANSWER_MODEL, [{"item_id": "M09", "prompt": build_prompt(item["question"], hits)}],
                    out_dir=ROOT / "outputs" / f"10_kb_{tag}",
                    prebaked_dir=PREBAKED.parent / f"10_kb_{tag}", verbose=False)[0]
    KB[tag] = " ".join((rec["output"] or rec["error"] or "").split())
    print(f"--- {tag} extraction\n{KB[tag]}\n")

if BLANK in permit_line(READ_CHUNKS):
    print("This extraction read the blank correctly, so both answers are right.\n"
          "That is this model on this form, not a property of the pipeline: a weaker extractor\n"
          "puts a permit number in that chunk, and the answer below it changes with no other warning.")
else:
    print("This extraction invented a permit number, and the answer above has just turned it\n"
          "into a cited fact.")

What permit to work number is recorded on work order WO-2026-05111? 

  M09: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
--- perfect extraction
The permit to work number recorded on work order WO-2026-05111 is blank on the form. [images/wo_10.png]

  M09: live call failed (TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client), using prebaked
--- actual extraction
The permit to work number recorded on work order WO-2026-05111 is blank on the form. [images/wo_10.png]

This extraction read the blank correctly, so both answers are right.
That is this model on this form, not a property of the pipeline: a weaker extractor
puts a permit number in that chunk, and the answer below it changes with no other warning.


If the extraction invented a permit number, the answer above quotes it, cites an image and reads exactly like the correct one. Nothing downstream can tell the difference: the format is right, the value is plausible, the citation resolves, and the regex validation from lab 09 passes it.

The only thing that disagrees is the image.

## 10. Verify against the pixels

Text retrieval is cheap and runs on every question. A vision call is expensive and runs on almost none. So spend it where being wrong is expensive: send the top-ranked image back to the vision model and ask about the one field the answer depends on.

This is the narrow single-field question from lab 09, used as a check rather than as an extraction.

In [25]:
VERIFY = """Look only at the field labelled "Permit to work no" on this form.
If nothing is written in it, answer exactly: EMPTY
Otherwise answer with exactly the text written in it, and nothing else."""

# Verify what the answer actually leaned on: the image it cited.
cited = [c for c in re.findall(r"\[([^\]]+)\]", KB["actual"]) if c.startswith("images/")]
target = cited[0] if cited else "images/wo_10.png"
print("the answer cites:", cited or "no image, so falling back to the form the question names")

indexed = permit_line(READ_CHUNKS).split(": ", 1)[-1]
if ANSWER_MODEL.backend == "prebaked":
    print("prebaked mode: no live model available to look at the image.")
else:
    check = ask(ANSWER_MODEL, VERIFY, image=ROOT / "corpus" / target)
    seen = (check["output"] or check["error"] or "").strip()
    print(f"\n{target}")
    print(f"  the index says : {indexed}")
    print(f"  the image says : {seen}")
    if seen.upper().startswith("EMPTY"):
        verdict = "agrees" if indexed == BLANK else "DISAGREES: the index holds a value the form does not"
    else:
        verdict = "agrees" if seen == indexed else f"DISAGREES: the form reads {seen}"
    print(" ", verdict)

the answer cites: ['images/wo_10.png']

images/wo_10.png
  the index says : (blank on the form)
  the image says : TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client` but got <class 'httpx2.Client'>
  DISAGREES: the form reads TypeError: Invalid `http_client` argument; Expected an instance of `httpx.Client` but got <class 'httpx2.Client'>


That is the whole pattern:

| Stage | Runs on | Cost |
|---|---|---|
| Extraction | every image, once, offline | the expensive part, paid when the image is filed |
| Retrieval | every question | milliseconds |
| Verification | the few fields that carry risk | one image call, on demand |

Which fields carry risk is not a modelling question. It is the list you already wrote in lab 09: permits, sign-offs, isolation references, anything that authorises work. Verify those and trust the index for the rest.

## 11. Try it: the same form, indexed three times (if you have time)

`wo_01`, `wo_04` and `wo_07` are one work order at three image qualities. In a real archive that is normal: the clean original, the scan somebody filed, and the fax a contractor sent back. All three get indexed, and their extractions disagree.

Add the degraded copies and ask the same question again. Which one ranks first, and does the number change?

In [26]:
COPIES = ["wo_01", "wo_04", "wo_07"]  # one work order: clean, scanned, faxed
extra = [i for i in COPIES if i not in INDEXED]
dupes = (read_extractions(EXTRACTOR, extra) if EXTRACTOR
         else {i: json.loads((TRUTH / f"{i}.json").read_text()) for i in extra})
ARCHIVE = ACTUAL.copy().add(image_chunks(dupes, EXTRACTOR or "ground truth"))

item = next(i for i in EVAL if i["id"] == "M04")
print(item["question"], "\n")
ranked = route(ARCHIVE)["drawings"].search(item["question"], k=len(ARCHIVE.chunks))
for rank, hit in enumerate(ranked, 1):
    if hit.doc_id in COPIES:
        reading = next((ln for ln in hit.text.splitlines() if ln.startswith("- Vibration")), "-")
        print(f"  drawing rank {rank}  {hit.source:22s} {reading}")
print("\nWhere the three disagree, retrieval rank decides which number the answer quotes.")
print("The fix is upstream: one record per work order, extracted from the best image available,")
print("not one chunk per file that happens to be in the archive.")

What vibration reading is recorded on work order WO-2026-04817? 

  drawing rank 1  images/wo_04.png       - Vibration: 4.6 mm/s
  drawing rank 2  images/wo_01.png       - Vibration: 4.6 mm/s
  drawing rank 4  images/wo_07.png       - Vibration: 3.6 mm/s

Where the three disagree, retrieval rank decides which number the answer quotes.
The fix is upstream: one record per work order, extracted from the best image available,
not one chunk per file that happens to be in the archive.


## 12. What to take away

- **Images join a text pipeline as extractions, not as images.** Extract once when the file is filed, index the structured result, and keep the image id in the chunk so every answer points at something a person can open.
- **Adding a source type changes the retrieval, not just the index.** Unlike things compete badly. Give each source type its own slots.
- **Retrieval accuracy is capped by extraction accuracy.** The tier scores from labs 08 and 09 are the ceiling for everything built on top of them.
- **An invented value becomes a cited answer.** Retrieval launders it: the format is right, the citation resolves, the validation rules pass. Only the pixels disagree.
- **Verify narrowly.** Re-read the image for the few fields that authorise work or money. That is affordable. Re-reading everything is not.

## Facilitator: save this run as the room's fallback

In [27]:
from vision_client import promote_to_prebaked

PROMOTE = False  # facilitator only: after a good live run, keep it for when the network or a model fails
if PROMOTE and RUN_MODE == "live":
    promote_to_prebaked(OUT, PREBAKED)